# Optional real-data exercise: two evaluation tasks on red wine

This optional extension adds model training after the synthetic delivery-metrics lab. It is not a prerequisite for understanding the metrics.

Use the supplied **`winequality-red.csv`** from the [UCI Wine Quality dataset](https://archive.ics.uci.edu/dataset/186/wine+quality). It contains 1,599 red-wine records, 11 physicochemical inputs, and a sensory quality score. The dataset is attributed to P. Cortez, A. Cerdeira, F. Almeida, T. Matos, and J. Reis (2009), under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The original semicolon-separated CSV is preserved.

Two separate tasks use the same input columns:

1. **Regression:** predict the recorded quality score. This treats an ordinal human rating as a numerical target for an educational approximation. Equal differences between ratings need not represent equal differences in perceived quality.
2. **Classification:** predict whether the recorded quality is at least 7. This is a declared educational label, not an official universal definition of good wine. A positive label means **quality ≥ 7**.

The **label cutoff 7** defines the event. A separate **decision threshold 0.50** turns a model score into a predicted positive. Do not confuse these numbers.


## Before running

Use the same Python packages as the core notebook. Place `winequality-red.csv` beside this notebook, or in a neighboring `data/` folder. In Colab, upload the CSV with the Files sidebar before running. This notebook never downloads data automatically.

Exact duplicate records are removed before splitting, to keep identical complete records from appearing on both sides of evaluation. We use a fixed 75%/25% train/test split stratified by the binary label, fit models only on training inputs, and evaluate on held-out data. The test set is not used to select a threshold or tune a model.

Both models are random forests. Their decision trees do not require standard scaling, so the original input units are retained. Understanding how a forest is trained is not a prerequisite for interpreting the metrics.

Do not expect this observational benchmark to establish future commercial performance or causal relationships.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve, r2_score,
)
from sklearn.model_selection import train_test_split

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

candidates = [
    Path("winequality-red.csv"),
    Path("data/winequality-red.csv"),
    Path("../data/winequality-red.csv"),
    Path("../public-data/winequality-red.csv"),
]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Upload winequality-red.csv beside this notebook, then rerun. "
        "The optional exercise does not download files."
    )

structured = np.genfromtxt(data_path, delimiter=";", names=True, dtype=float)
column_names = list(structured.dtype.names)
records = np.column_stack([structured[name] for name in column_names])
assert records.shape == (1599, 12)
assert column_names[-1] == "quality"
assert np.isfinite(records).all()
print(f"Loaded {len(records)} records with {len(column_names) - 1} inputs.")


In [ ]:
# Retain the first occurrence of each complete record, including its target.
_, first_indices = np.unique(records, axis=0, return_index=True)
unique_records = records[np.sort(first_indices)]
removed = len(records) - len(unique_records)

feature_names = column_names[:-1]
X = unique_records[:, :-1]
quality = unique_records[:, -1]
y_binary = (quality >= 7).astype(int)
assert "quality" not in feature_names
assert X.shape[1] == 11

train_indices, test_indices = train_test_split(
    np.arange(len(X)), test_size=0.25, random_state=42, stratify=y_binary,
)
X_train, X_test = X[train_indices], X[test_indices]
class_train, class_test = y_binary[train_indices], y_binary[test_indices]
quality_train, quality_test = quality[train_indices], quality[test_indices]

assert set(train_indices).isdisjoint(set(test_indices))
assert len(train_indices) + len(test_indices) == len(unique_records)
print(f"Removed {removed} exact duplicate records; retained {len(unique_records)}.")
print(f"Training records: {len(train_indices)}; test records: {len(test_indices)}.")
print(f"Positive prevalence: train {class_train.mean():.1%}, test {class_test.mean():.1%}.")


## Predict first: compare with a majority-class baseline

If relatively few test wines have quality ≥ 7, an always-negative classifier may have high accuracy. Will the fitted classifier find any positive cases? Does finding more positives necessarily mean every positive prediction is trustworthy?

The classifier learns only from training inputs and labels, then scores the held-out inputs. The numbers below evaluate the model; the forest's training procedure is outside the main metrics lesson. Its settings are fixed in advance, not tuned on the test set.


In [ ]:
classifier = RandomForestClassifier(
    n_estimators=150, max_depth=6, min_samples_leaf=4,
    random_state=42, n_jobs=1,
)
classifier.fit(X_train, class_train)
quality_scores = classifier.predict_proba(X_test)[:, list(classifier.classes_).index(1)]
# These are model probability estimates, without a calibration assessment here.
assert np.isfinite(quality_scores).all()
assert np.all((quality_scores >= 0) & (quality_scores <= 1))
decision_threshold = 0.50
class_predictions = (quality_scores >= decision_threshold).astype(int)

dummy_classifier = DummyClassifier(strategy="most_frequent").fit(X_train, class_train)
dummy_predictions = dummy_classifier.predict(X_test)

print("Positive event: quality >= 7; score decision threshold: 0.50")
print("Model          Accuracy Precision Recall    F1")
for label, guessed in [("Majority", dummy_predictions), ("Forest", class_predictions)]:
    print(f"{label:14s} {accuracy_score(class_test, guessed):.3f}    "
          f"{precision_score(class_test, guessed, zero_division=0):.3f}     "
          f"{recall_score(class_test, guessed, zero_division=0):.3f}  "
          f"{f1_score(class_test, guessed, zero_division=0):.3f}")

cm = confusion_matrix(class_test, class_predictions, labels=[1, 0])
tp, fn, fp, tn = cm.ravel()
print("Confusion matrix, rows actual [P=1,N=0], columns predicted [P=1,N=0]:")
print(cm)
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"ROC-AUC from full scores: {roc_auc_score(class_test, quality_scores):.4f}")
assert int(cm.sum()) == len(test_indices)


**Interpretation checkpoint:** Precision is the fraction of predicted quality≥7 wines whose recorded rating was indeed ≥7. Recall is the fraction of all recorded quality≥7 wines that were predicted positive. The matrix uses P before N on both axes: `[[TP, FN], [FP, TN]]`. Unpacking is `tp, fn, fp, tn = matrix.ravel()`. The baseline's zero precision uses `zero_division=0` as a reporting convention when it predicts no positives.

The 0.50 threshold is fixed for this exercise. Explore threshold changes later on a separate validation split; do not select the best-looking threshold on this final test set and report that same test result as an unbiased assessment.


In [ ]:
fpr, tpr, _ = roc_curve(class_test, quality_scores, pos_label=1)
fig, ax = plt.subplots(figsize=(6, 4.8))
ax.plot(fpr, tpr, color="#007f78", label=f"Held-out AUC = {roc_auc_score(class_test, quality_scores):.3f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="AUC=0.5 reference")
ax.set(xlabel="False positive rate", ylabel="Recall (true positive rate)",
       title="Ranking quality on held-out red-wine records")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## A numerical quality predictor and its baseline

The regression task predicts the recorded quality score directly. A separate random forest regressor learns from the same training indices. A `DummyRegressor` learns the **training-set mean** as an honest deployable baseline. Standard test R² uses the **test-target mean** in its denominator, which is a metric convention, so the training-mean dummy can have slightly negative test R².

**Predict first:** Is positive R² guaranteed on held-out data? Can a small numerical error still matter near the classification label cutoff?


In [ ]:
regressor = RandomForestRegressor(
    n_estimators=150, max_depth=6, min_samples_leaf=4,
    random_state=42, n_jobs=1,
)
regressor.fit(X_train, quality_train)
quality_predictions = regressor.predict(X_test)
assert np.isfinite(quality_predictions).all()
dummy_regressor = DummyRegressor(strategy="mean").fit(X_train, quality_train)
dummy_quality_predictions = dummy_regressor.predict(X_test)

model_sse = float(np.sum((quality_test - quality_predictions) ** 2))
test_mean_sse = float(np.sum((quality_test - quality_test.mean()) ** 2))
manual_r2 = 1 - model_sse / test_mean_sse
library_r2 = r2_score(quality_test, quality_predictions)
np.testing.assert_allclose(manual_r2, library_r2)

print(f"Forest test R2: {library_r2:.4f}")
print(f"Training-mean dummy test R2: {r2_score(quality_test, dummy_quality_predictions):.4f}")
print(f"Test-target mean: {quality_test.mean():.4f}; training mean: {quality_train.mean():.4f}")
print(f"R2 = 1 - {model_sse:.3f}/{test_mean_sse:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(quality_test, quality_predictions, s=18, alpha=0.4, color="#15324f")
ax.plot([3, 8], [3, 8], color="#007f78", linestyle="--", label="Exact numerical prediction")
ax.set(xlabel="Recorded quality score", ylabel="Predicted quality score",
       title="Regression predictions on held-out records")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## What to explain after running

- Report the full confusion matrix alongside the threshold and positive-class definition. Accuracy, precision, recall, and F1 answer different questions.
- ROC-AUC uses the score ordering, not the selected 0.50 decision alone. It does not check calibration.
- R² compares squared numerical errors with the test-mean benchmark. Test performance can be negative.
- The sensory score is ordinal and the chosen quality≥7 event is educational. The two tasks have different targets and evaluation questions.
- Exact-record deduplication reduces a simple leakage risk. A single fixed random split does not establish robustness across producers, batches, regions, time, or other deployment shifts.

No benchmark score here is a tuning target. The exercise demonstrates a complete training/evaluation separation and how to interpret the resulting metrics.
